<center><h1 style="margin-bottom: 0px;">Artificial Intelligence (25/26)</h1></center>
<center><h2 style="margin-top: 0px;">Adversarial Search Strategies and Decision Trees</h2></center>

#### <center> Diogo Sousa (202306336), Joana Antunes (202405702), Sílvia Pinto (202405988) </center> <br>

In [3]:
import copy
import time
import random
import os
from gameplay_functions import *

### **1. Introduction** <br>
<div style="text-align: justify;"><p style="text-indent: 2em;">This report details the development and implementation of a program designed to master PopOut, a dynamic variant of the classic Connect-4 game. Unlike traditional gameplay, PopOut introduces a "pop" mechanic where players can remove their own discs from the bottom row, shifting the entire column downward and fundamentally altering the board state. 
<p style="text-indent: 2em;">The primary objective of this assignment is to explore and implement adversarial search for the Pop Out game, and learning algorithms to play it. To achieve this, we implemented Monte Carlo Tree Search (MCTS) and Decision Trees to power the decision-making process. MCTS was implemented using Upper Confidence Bound for Trees (UCT) as an evaluation function to balance exploration and exploitation during move selection. Decision Trees were implemented using the ID3 procedure and a custom dataset generated by our MCTS implementation, mapping specific game states to optimal moves. 
<p style="text-indent: 2em;">These search algorithms allowed us to develop three distinct gameplay modes: Human vs. Human, Human vs. Computer and Computer vs. Computer (pitting different algorithms against one another). Adhering to the project guidelines, all core logic for the decision trees was implemented from scratch without the use of automated libraries like scikit-learn. 
<p style="text-indent: 2em;">This report documents our game implementation, technical decisions, strategies used to optimize our search algorithms and an analysis of the results obtained across both search and classification tasks.

### **2. Pop Out - Game Rules** <br>
<div style="text-align: justify;"><p style="text-indent: 2em;">There are three types of moves in Pop Out:

> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**1.** *Insert piece (I): Places a piece at the top of a column, which then falls until the lowest empty position in that column.*

> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**2.** *Remove piece (R): Removes the base piece of a column, causing every piece above it to fall one position.*

> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**3.** *Draw (D): Declares a tie - only if the board is completely full, or the current game state has been repreated three or more times.*

<p style="text-indent: 2em;">The goal of the game is to make a 4-in-row - either horizontally, vertically or diagonally.
<p style="text-indent: 2em;">IMPORTANT RULE: If a "remove" play causes a 4-in-row for both players, the winner is the player who made that "remove".</div>

### **3. Pop Out - Implementation** <br>
<div style="text-align: justify;"><p style="text-indent: 2em;">The core part of our implementation is the class Pop Out, which represents an instance of a Pop Out game. It has a board, which starts as an empty 7 by 6 grid, a current player (starts as "O" by default), a board history to register the number of times we have been in a certain state of the board, the winner of the game (starts as None, but will eventually become defined when the game ends) and the state of the game (is it finished or still running).</div>

In [4]:
class Pop_Out():                                                    # The class which represents an instance of our Pop Out game.
    def __init__(self, board = None, cur_player = "O", board_history = None, winner = None, terminal = False):
        # An instance of the game has a board, the current player, the history of previous moves and the winner of the game (if it is finished).
        self.board = copy.deepcopy(board) if board else [["."] * 7 for _ in range(6)]
        self.cur_player = cur_player
        #We need the board history to know if we have already been in a certain state more than 3 times, in order to allow players to call a draw.
        self.board_history = copy.deepcopy(board_history) if board_history else {self.board_to_string(): 1}
        self.winner = winner
        self.terminal = terminal 

<div style="text-align: justify;"><p style="text-indent: 2em;">The following functions define a few simple actions that will be helpful for other operations on our game. Respectively, board_to_string converts a matrix board into a printable string which can be used in our text interface; change_player allows to change players between each play; board_is_full checks if the board has every single position filled; register_state adds the current state to the board_history.</div>

In [5]:
def board_to_string(self):                                          # Turns the board into a string, in order to be printable and readable for text interface.
    s = ""
    for row in self.board: s += " ".join(row) + "\n"
    return s

Pop_Out.board_to_string = board_to_string

def change_player(self):                                            # Switches players
    self.cur_player = "X" if self.cur_player == "O" else "O"

Pop_Out.change_player = change_player
    
def board_is_full(self):                                            # Checks if the board is completely full (in which case the draw is playable)
    return all(cell != "." for cell in self.board[0])

Pop_Out.board_is_full = board_is_full

def register_state(self):                                           # Uploads the state in the board history.
    key = self.board_to_string()
    if key in self.board_history:                                   # If it has already been visited, add 1 to its number of visits.
        self.board_history[key] += 1
    else:
        self.board_history[key] = 1                                 # Else, mark it as visited once.

Pop_Out.register_state = register_state

<div style="text-align: justify;"><p style="text-indent: 2em;">Then, we have implemented the functions for "insert" and "remove" operations. Since a move made by a human player can be invalid for a certain state of the game, we have created the invalid_move function to handle those cases and guide the player; it is available in the file gameplay_functions.py, as well as the clear_terminal function.</div>

In [6]:
def insert(self, col, printing):                                    # Performs an "insert" move.
    c = col - 1
    if self.board[0][c] != ".": invalid_move(self)                  # If the column is full already, declare the move invalid.
    self.board[0][c] = self.cur_player                              # Else, insert current player's piece at the top...
    row = 0
    while row < 5 and self.board[row + 1][c] == ".":                # and let it fall until it lands on another piece, or the end of the board.
        if (printing):                                              # Produces the animation of the fall, which will not be seen during searching phase of MCTS.
            clear_output(wait = True)
            print(self.board_to_string())
            time.sleep(0.5)
        self.board[row][c] = "."
        self.board[row + 1][c] = self.cur_player
        row += 1
    if (printing): clear_output(wait = True)
    self.update_terminal_status()                                   # Checks if this "insert" has produced a win...
    if not self.terminal:                                           # If not, change players and register this visit to the board.
        self.change_player()
        self.register_state()

Pop_Out.insert = insert

def remove(self, col, printing):                                    # Performs a "remove" move.
    c = col - 1
    if self.board[5][c] != self.cur_player: 
        invalid_move(self)                                          # If the bottom piece does not belong to the player, declare move as invalid.
        return
    for row in range(5, 0, -1):                                     # Else, remove piece and let the ones above it fall down 1 place.
        if (printing):                                              # Produces the animation of the fall, which will not be seen during searching phase of MCTS.
            clear_output(wait = True)
            print(self.board_to_string())
            time.sleep(0.5)
        self.board[row][c] = self.board[row - 1][c]
    self.board[0][c] = "."
    if (printing): clear_output(wait = True)
    self.update_terminal_status()                                   # Checks if this "remove" has produced a win...
    if not self.terminal:                                           # If not, change players and register this visit to the board.
        self.change_player()
        self.register_state()

Pop_Out.remove = remove

<div style="text-align: justify;"><p style="text-indent: 2em;">Next, we have the functions which check if a player has made a 4-in-row (function has_four) and which evaluate if the game has ended based on the current 4-in-rows in the board (update_terminal_status). The function check is responsible for finding 4-in-rows in each line, column and diagonal, and it is available in file gameplay_functions.py. Lastly, the function apply_move is responsible for executing a given move, depending on its kind - insert, remove, draw.</div>

In [7]:
def has_four(self, player):                                         # Function that checks if a player has made a 4-in-row in the current game state.
    for row in range(6):                                            # Checks each row first,
        if check(self.board[row], player): return True

    for col in range(7):                                            # then each column,
        cur_col = []
        for row in range(6):
            cur_col.append(self.board[row][col])
        if check(cur_col, player): return True

    for row in range(3):                                            # then the diagonals.
        for col in range(4):
            diag = []
            diag2 = []
            for i in range(4):
                diag.append(self.board[row + i][col + i])
                diag2.append(self.board[row + 3 - i][col + i])
            if check(diag, player) or check(diag2, player): return True
    return False

Pop_Out.has_four = has_four

def update_terminal_status(self):                                   # Checks if the game has ended, in its current state.
    o_wins = self.has_four("O")
    x_wins = self.has_four("X")

    if not o_wins and not x_wins: return                            # If no one has won yet, things remain the same.
    self.terminal = True                                            # If someone has won. then the game has terminated.
    if o_wins and x_wins: self.winner = self.cur_player             # If both players win, we are in the presence of a "remove", so the current player wins.
    elif o_wins: self.winner = "O"                                  # Else, if only O wins, declare it the winner.
    elif x_wins: self.winner = "X"                                  # Lastly, if only X wins, declare it the winner.

Pop_Out.update_terminal_status = update_terminal_status

def apply_move(self, move, printing = True):                        # Reads a given move and executes the corresponding action. 
    kind, value = move[0], move[1]
    if kind == "I":                                                 # Perform "insert"
        self.insert(value, printing)                            
    elif kind == "R":                                               # Perform "remove"
        self.remove(value, printing)                            
    elif kind == "D":                                               # Perform "draw"...
        if self.board_is_full() or self.board_history.get(self.board_to_string(), 0) >= 3:
            self.winner = None
            self.terminal = True                                    # and end the game,
        else: invalid_move(self)                                    # unless the conditions for a draw have not been met.

Pop_Out.apply_move = apply_move

<div style="text-align: justify;"><p style="text-indent: 2em;">Finally, we decided to create a separate class CVP_Pop_Out, which extends Pop_Out, to allow for a switch between player types (human and computer). This is made by overwriting the two functions which handle player status. The initialization function uses the exact same structure as Pop_Out, but adds a new variable "type_player" which indicates whether the current player is "human" or "computer". The starting player is randomized by the function "randomize_player", present in the gameplay_functions.py file. Then, when there is a change in players between moves, the function change_player also shifts the player type from "human" to "computer" and vice-versa.</div>

In [8]:
class CVP_Pop_Out(Pop_Out):                                         # Extends Pop Out game by changing a couple of its functions...
    def __init__(self, board = None, cur_player = "O", board_history = None, winner = None, terminal = False, start_player = None):
        super().__init__(board, cur_player, board_history, winner, terminal)
        self.type_player = randomize_player(start_player)           # to allow for a human to play against a computer,

    def change_player(self):                                        # and to change the player type according to that.
        super().change_player()
        self.type_player = "human" if self.type_player == "computer" else "computer"

### **4. Monte Carlo Tree Search** <br>
<div style="text-align: justify;"><p style="text-indent: 2em;">We proceeded to create various versions of Monte Carlo Tree Search (MCTS) algorithm. It is based on the execution of multiple random simulations (playouts) starting from a given state of the game. For each possible move:</div>

> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**1.** *That move is applied to the current game state;*

> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**2.** *From that point on, the game is simulated in a random way;*

> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**3.** *The final result of that random simulation is registered (victory, defeat or draw);*

> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**4.** *The process described in points 2 and 3 is repeated multiple times, and the mean of the results is calculated.*

<div style="text-align: justify;"><p style="text-indent: 2em;">The chosen move is the one with the best mean result.</div><br>

<div style="text-align: justify;"><p style="text-indent: 2em;">Before implementing MCTS, we needed to add a few more functions to our Pop_Out class. These will be used in our Monte Carlo algorithm and, respectively, they allow us to clone a game by creating a new one with the exact same characteristics (function clone), return the valid moves for a current game state (function get_valid_moves) and, finally, return the result of our game in the form of a "reward" for a simulation of MCTS (function get_result). </div>

In [9]:
def clone(self):                                                    # Function that clones a Pop Out game, to be used for Monte Carlo Tree Search.
    return Pop_Out(board = self.board,                              # Simply creates a new game with all of the exact characteristics of the previous one.
                cur_player = self.cur_player, 
                board_history = self.board_history, 
                winner = self.winner, 
                terminal = self.terminal)  

Pop_Out.clone = clone

def get_valid_moves(self):                                          # For the computer to select its best move, shows it what moves are available.
    moves = []
    for col in range(7):                                        
        if self.board[0][col] == ".":                               # In each column, checks if there is space for an insert at the top.
            moves.append(("I", col + 1))
        if self.board[5][col] == self.cur_player:                   # Also, checks if the last cell belongs to the current player; if so, it is removable.
            moves.append(("R", col + 1))

    if self.board_is_full() or self.board_history.get(self.board_to_string(), 0) >= 3: 
        moves.append(("D", None))                                   # If the board is full, or the current state has been repeated 3 or more times, then draw is available.
    return moves

Pop_Out.get_valid_moves = get_valid_moves

def get_result(self, player):                                       # Returns the result of the game as a "reward" value for MCTS to evaluate the path.
    if self.winner is None: return 0.5
    if self.winner == player: return 1.0
    return 0.0

Pop_Out.get_result = get_result

<div style="text-align: justify;"><p style="text-indent: 2em;">The following algorithm is the simplest form of MCTS (also known as "Flat Monte Carlo"). Its goal is to select the best move with the help of random game simulations. The mean score of the simulations reflects the expected quality of the move, in order to select the best play. Even if it is a simple version, it allows us to make informed decisions instead of a purely random strategy. However, it presents a few limitations, such as not reusing information between simulations and having a strong dependency on the number of simulations executed, which can impact execution time.</div>

In [ ]:
# Import the Pop_Out class, which represents the game state and logic
from pop_out import Pop_Out
# Create a new game state (initial empty board, starting player...)
state = Pop_Out()

In [ ]:
#Fist version: Flart monte carlo

def random_playout(simulation_state):
    #Selects a random legal move from the current simulated state
    return random.choice(simulation_state.get_valid_moves())


def monte_carlo_move(state, simulations_per_move=50, playout_func=random_playout):
    #Stores the player for whom we are choosing the mode
    root_player = state.cur_player
    #Variables used to keep track of the best move found so far
    best_move = None
    best_score = -1

    #Test every legal move from the current state
    for candidate_move in state.get_valid_moves():
        total_score = 0
        #run several simulations for this candidate move
        for _ in range(simulations_per_move):
            #Clone the state so that the real game is not changed
            simulation_state = state.clone()
            #Apply the candidate move as the first move of the simulation 
            simulation_state.apply_move(candidate_move, printing=False)
 
           #Continue the game randonmly until it reaches a terminal state
            while not simulation_state.terminal:
                playout_move = playout_func(simulation_state)
                simulation_state.apply_move(playout_move, printing=False)
            #Add the result of this simulation from the root player's perspective
            total_score += simulation_state.get_result(root_player)
        #Calculate the average score
        average_score = total_score / simulations_per_move

        #If this move is better than the last one store it 
        if average_score > best_score:
            best_score = average_score
            best_move = candidate_move
   
    return best_move

We can call this version using the next piece of code

In [18]:
move = monte_carlo_move(state, simulations_per_move=50)

Second version

We decided to implement a second version, where we improve the last Monte Carlo algorithm by introducting a heuristic-based simulation policy. Instead of relying entirely on random playouts, the agent incorporates simple strategic rules during the simulation phase. In particular, it prioritizes moves that lead to an immediate win and avoids moves that allow the opponent to win in the next turn. When no such situations are detected, the agent falls back to random move selection.

This modification results in more realistic and informative simulations, as the playouts better reflect plausible gameplay rather than purely random behavior. Consequently, the algorithm is able to produce more accurate evaluations of candidate moves and generally achieves better performance than the basic version. However, despite this improvement, the method still evaluates each move independently and does not reuse information between simulations, meaning it remains less efficient than more advanced approaches such as Monte Carlo Tree Search (MCTS).


In [19]:
# Version 2 — Heuristic-based playout for Monte Carlo

def heuristic_playout(simulation_state):
    # Uses the heuristic-guided move instead of a random move
    return heuristic_guided_move(simulation_state)


def heuristic_guided_move(state):
    # List to store moves that do not immediately lose the game
    safe_moves = []

    # Iterate through all possible valid moves
    for move in state.get_valid_moves():
        clone = state.clone()
        clone.apply_move(move, printing=False)

        # If this move wins immediately, play it
        if clone.terminal and clone.winner == state.cur_player:
            return move

        # Check whether the opponent can win immediately after this move
        opponent_can_win = False

        for opponent_move in clone.get_valid_moves():
            temp = clone.clone()
            temp.apply_move(opponent_move, printing=False)

            # If the opponent can win immediately after this move, mark it as unsafe
            if temp.terminal and temp.winner == clone.cur_player:
                opponent_can_win = True
                break
        #  If the opponent cannot win immediately, consider this move safe
        if not opponent_can_win:
            safe_moves.append(move)
# If there are safe moves available, choose one randomly
    if safe_moves:
        return random.choice(safe_moves)
#  If no safe moves exist, fall back to a completely random move
    return random.choice(state.get_valid_moves())


The main difference between the two versions of the Monte Carlo algorithm lies in the simulation strategy used during the playout phase. In the first version, simulations are entirely random, meaning that each move is selected uniformly from the set of valid moves. While this approach is simple and easy to implement, it often produces unrealistic game scenarios and less reliable evaluations. In contrast, the second version introduces a heuristic-based simulation policy, where the agent prioritizes moves that result in an immediate win and avoids moves that allow the opponent to win in the next turn. When no such situations are detected, the algorithm falls back to random selection among safe moves. This results in more realistic simulations and improves the overall quality of decision-making, while still maintaining the simplicity of the original Monte Carlo approach.

After this two simpler initials aproaches, we are now starting to implement the real monte carlo algorithm 

## Criação do data-set

Geramos o data-set do jogo, deixando as várias versões do algoritmo MCT realizar vários jogos, escolhemos os jogos que faziam mais sentido em termos de jogadas e resultados finais e evitamos jogos repetidos. Depois disso guardamos como data-set as todas as jogadas e estados do tabuleiro para cada um desses jogos que selecionamos anteriormente. (Não tenho a certeza acho que foi assim que o professor explicou)



# Árvores de decisão

(para o iris data-set e o do jogo)

In [10]:
import pandas as pd

df = pd.read_csv("iris.csv")

In [11]:
if (os.name == "nt"):
    os.system('start cmd /k python MAIN.py')
else: os.system("osascript -e 'tell application \"Terminal\" to do script \"python3 " + os.getcwd() + "/MAIN.py\"'")